In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# Ethnicity
# To download original dataset go to - 
# https://www.ons.gov.uk/datasets/TS021/editions/2021/versions/3
ethnicity_csv = r"C:\Users\abhimanya.achara\Downloads\TS021-2021-3-filtered-2025-02-06T08_07_15Z.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 2.5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframe
lsoa_ethnicity_df = pd.read_csv(ethnicity_csv)
lsoa_ethnicity_df.head()

,Lower layer Super Output Areas Code,Lower layer Super Output Areas,Ethnic group (20 categories) Code,Ethnic group (20 categories),Observation
0,E01000001,City of London 001A,-8,Does not apply,0
1,E01000001,City of London 001A,1,"Asian, Asian British or Asian Welsh: Bangladeshi",4
2,E01000001,City of London 001A,2,"Asian, Asian British or Asian Welsh: Chinese",66
3,E01000001,City of London 001A,3,"Asian, Asian British or Asian Welsh: Indian",49
4,E01000001,City of London 001A,4,"Asian, Asian British or Asian Welsh: Pakistani",2


In [6]:
#create pivot table
lsoa_ethincity_df_P = pd.pivot_table(lsoa_ethnicity_df, values='Observation', index=['Lower layer Super Output Areas Code'], columns=['Ethnic group (20 categories)'], aggfunc=np.sum)
lsoa_ethincity_df_P = lsoa_ethincity_df_P.reset_index()

#drop columns
lsoa_ethincity_df_P.drop(['Does not apply'], 1, inplace=True)

# Dictionary for renaming columns with corrected keys
column_rename_map = {
    "Asian, Asian British or Asian Welsh: Bangladeshi": "asian_bangladeshi",
    "Asian, Asian British or Asian Welsh: Chinese": "asian_chinese",
    "Asian, Asian British or Asian Welsh: Indian": "asian_indian",
    "Asian, Asian British or Asian Welsh: Pakistani": "asian_pakistani",
    "Asian, Asian British or Asian Welsh: Other Asian": "asian_other",
    
    "Black, Black British, Black Welsh, Caribbean or African: African": "black_african",
    "Black, Black British, Black Welsh, Caribbean or African: Caribbean": "black_caribbean",
    "Black, Black British, Black Welsh, Caribbean or African: Other Black": "black_other",
    
    "Mixed or Multiple ethnic groups: Other Mixed or Multiple ethnic groups": "mixed_or_multiple_other",
    "Mixed or Multiple ethnic groups: White and Asian": "mixed_or_multiple_white_and_asian",
    "Mixed or Multiple ethnic groups: White and Black African": "mixed_or_multiple_white_and_black_african",
    "Mixed or Multiple ethnic groups: White and Black Caribbean": "mixed_or_multiple_white_and_black_caribbean",
    
    "Other ethnic group: Any other ethnic group": "other_other",
    "Other ethnic group: Arab": "other_arab",
    
    "White: English, Welsh, Scottish, Northern Irish or British": "white_british",
    "White: Gypsy or Irish Traveller": "white_gypsy_or_irish",
    "White: Irish": "white_irish",
    "White: Other White": "white_other",
    "White: Roma": "white_roma"
}

# Rename columns using the dictionary
lsoa_ethincity_df_P.rename(columns=column_rename_map, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_ethincity_df_P.columns = (
    lsoa_ethincity_df_P.columns
    .str.cat(['_count'] * len(lsoa_ethincity_df_P.columns))  # Append '_count' to each column
)

#rename columns
lsoa_ethincity_df_P.rename(columns = {'Lower layer Super Output Areas Code_count':'lsoa21cd'},inplace = True)
# Display the updated DataFrame

lsoa_ethincity_df_P['total_ethnicity_pop'] = lsoa_ethincity_df_P.sum(axis=1,numeric_only=True)

lsoa_ethincity_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_16224\1350218462.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_ethincity_df_P.drop(['Does not apply'], 1, inplace=True)


Ethnic group (20 categories),lsoa21cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop
0,E01000001,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474
1,E01000002,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386
2,E01000003,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612
3,E01000005,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101
4,E01000006,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1845


## 2. Feature Engineering

In [7]:
# Create percentage columns for detailed ethnicity
total_ethnicity_pop = 'total_ethnicity_pop'
for col in lsoa_ethincity_df_P.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    lsoa_ethincity_df_P[perc_col] = (lsoa_ethincity_df_P[col] / lsoa_ethincity_df_P[total_ethnicity_pop]) * 100

lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lsoa21cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc
0,E01000001,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269
1,E01000002,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900
2,E01000003,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139
3,E01000005,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306
4,E01000006,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1845,11.598916,0.596206,8.834688,4.769648,22.168022,6.937669,2.276423,1.626016,1.409214,0.867209,0.704607,0.813008,4.444444,0.162602,13.062331,0.0,0.813008,18.319783,0.596206


In [8]:
# Feature Engineering - Aggregating Ethnicity Counts
ethnicity_groups = {
    "asian_count": [
        "asian_bangladeshi_count",
        "asian_chinese_count",
        "asian_indian_count",
        "asian_pakistani_count",
        "asian_other_count"
    ],
    "black_count": [
        "black_african_count",
        "black_caribbean_count",
        "black_other_count"
    ],
    "mixed_or_multiple_count": [
        "mixed_or_multiple_other_count",
        "mixed_or_multiple_white_and_asian_count",
        "mixed_or_multiple_white_and_black_african_count",
        "mixed_or_multiple_white_and_black_caribbean_count"
    ],
    "other_ethnic_count": [
        "other_other_count",
        "other_arab_count"
    ],
    "white_count": [
        "white_british_count",
        "white_gypsy_or_irish_count",
        "white_irish_count",
        "white_other_count",
        "white_roma_count"
    ]
}

# Compute aggregated ethnicity counts
for new_col, cols in ethnicity_groups.items():
    lsoa_ethincity_df_P[new_col] = lsoa_ethincity_df_P[cols].sum(axis=1)

# Display results
lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lsoa21cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count
0,E01000001,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269,158,11,56,68,1181
1,E01000002,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900,181,11,60,50,1084
2,E01000003,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139,153,56,101,107,1195
3,E01000005,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306,354,119,79,125,424
4,E01000006,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1845,11.598916,0.596206,8.834688,4.769648,22.168022,6.937669,2.276423,1.626016,1.409214,0.867209,0.704607,0.813008,4.444444,0.162602,13.062331,0.0,0.813008,18.319783,0.596206,885,200,70,85,605


In [9]:
# Create percentage columns for simplified ethnicity
total_ethnicity_pop = 'total_ethnicity_pop'
for col in lsoa_ethincity_df_P.columns[-5:]:
    perc_col = col.replace('_count', '_perc')
    lsoa_ethincity_df_P[perc_col] = (lsoa_ethincity_df_P[col] / lsoa_ethincity_df_P[total_ethnicity_pop]) * 100

lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lsoa21cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc
0,E01000001,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269,158,11,56,68,1181,10.719132,0.746269,3.799186,4.613297,80.122117
1,E01000002,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900,181,11,60,50,1084,13.059163,0.793651,4.329004,3.607504,78.210678
2,E01000003,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139,153,56,101,107,1195,9.491315,3.473945,6.265509,6.637717,74.131514
3,E01000005,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306,354,119,79,125,424,32.152589,10.808356,7.175295,11.353315,38.510445
4,E01000006,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1845,11.598916,0.596206,8.834688,4.769648,22.168022,6.937669,2.276423,1.626016,1.409214,0.867209,0.704607,0.813008,4.444444,0.162602,13.062331,0.0,0.813008,18.319783,0.596206,885,200,70,85,605,47.967480,10.840108,3.794038,4.607046,32.791328


## 4. Load GIS shapefile 

In [10]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [11]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_16224\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [12]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_16224\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [13]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [14]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_16224\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [15]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [16]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_16224\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [17]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [18]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [19]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [21]:
census2021_ethnicity_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_ethincity_df_P, how = 'left', on='lsoa21cd')
census2021_ethnicity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269,158,11,56,68,1181,10.719132,0.746269,3.799186,4.613297,80.122117
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900,181,11,60,50,1084,13.059163,0.793651,4.329004,3.607504,78.210678
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139,153,56,101,107,1195,9.491315,3.473945,6.265509,6.637717,74.131514
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306,354,119,79,125,424,32.152589,10.808356,7.175295,11.353315,38.510445
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1845,11.598916,0.596206,8.834688,4.769648,22.168022,6.937669,2.2764

# 8. Final tweaks

In [25]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'wd22cd',
 'wd22nm',
 'lad22cd',
 'lad22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'asian_count',
 'black_count',
 'mixed_or_multiple_count',
 'other_ethnic_count',
 'white_count',
 'asian_perc',
 'black_perc',
 'mixed_or_multiple_perc',
 'other_ethnic_perc',
 'white_perc',
 'asian_bangladeshi_count',
 'asian_chinese_count',
 'asian_indian_count',
 'asian_other_count',
 'asian_pakistani_count',
 'black_african_count',
 'black_caribbean_count',
 'black_other_count',
 'mixed_or_multiple_other_count',
 'mixed_or_multiple_white_and_asian_count',
 'mixed_or_multiple_white_and_black_african_count',
 'mixed_or_multiple_white_and_black_caribbean_count',
 'other_other_count',
 'other_arab_count',
 'white_british_count',
 'white_gypsy_or_irish_count',
 'white_irish_count',
 'white_other_count',
 'white_roma_count',
 'total_ethnicity_pop',
 'asian_bangladeshi_perc',
 'asian_chinese_perc',
 'asian_indian_perc',
 'asian_other_perc',
 'asian_pakistani_perc',
 'black_african_perc',
 'black_caribbean_perc',
 'black_other_perc',
 'mixed_or_multiple_other_perc',
 'mixed_or_multiple_white_and_asian_perc',
 'mixed_or_multiple_white_and_black_african_perc',
 'mixed_or_multiple_white_and_black_caribbean_perc',
 'other_other_perc',
 'other_arab_perc',
 'white_british_perc',
 'white_gypsy_or_irish_perc',
 'white_irish_perc',
 'white_other_perc',
 'white_roma_perc'

]

census2021_ethnicity_lsoa2021_gdb_df = census2021_ethnicity_lsoa2021_gdb_df[final_column_order]

census2021_ethnicity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,158,11,56,68,1181,10.719132,0.746269,3.799186,4.613297,80.122117,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,181,11,60,50,1084,13.059163,0.793651,4.329004,3.607504,78.210678,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,153,56,101,107,1195,9.491315,3.473945,6.265509,6.637717,74.131514,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,354,119,79,125,424,32.152589,10.808356,7.175295,11.353315,38.510445,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E05014066,Northbury,E09000002,Barking and Dagenham,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,885,200,70,85,605,47.967480,10.840108,3.794038,4.607046,32.791328,214,11,163,88,409,128,42,30,26,16,13,15,82,3,241,0,15,338,11,1

# 8. Create dominant field

In [27]:
def determine_dominant_group(row):
    age_columns = [
        'asian_perc',
        'black_perc',
        'mixed_or_multiple_perc',
        'other_ethnic_perc',
        'white_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value*2)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_ethnicity_lsoa2021_gdb_df['dominant_ethnic_group'] = census2021_ethnicity_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_ethnicity_lsoa2021_gdb_df.head()

C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\geodataframe.py:1351: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,dominant_ethnic_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,158,11,56,68,1181,10.719132,0.746269,3.799186,4.613297,80.122117,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269,white
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,181,11,60,50,1084,13.059163,0.793651,4.329004,3.607504,78.210678,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900,white
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,153,56,101,107,1195,9.491315,3.473945,6.265509,6.637717,74.131514,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139,white
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,354,119,79,125,424,32.152589,10.808356,7.175295,11.353315,38.510445,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306,white
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E05014066,Northbury,E09000002,Barking and Dagenham,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,885,200,70,85,605,47.967480,10.840108,3.794038,4.607046,32.791328,214,11,163,88,40

In [29]:
def determine_dominant_group(row):
    age_columns = [
 'asian_bangladeshi_perc',
 'asian_chinese_perc',
 'asian_indian_perc',
 'asian_other_perc',
 'asian_pakistani_perc',
 'black_african_perc',
 'black_caribbean_perc',
 'black_other_perc',
 'mixed_or_multiple_other_perc',
 'mixed_or_multiple_white_and_asian_perc',
 'mixed_or_multiple_white_and_black_african_perc',
 'mixed_or_multiple_white_and_black_caribbean_perc',
 'other_other_perc',
 'other_arab_perc',
 'white_british_perc',
 'white_gypsy_or_irish_perc',
 'white_irish_perc',
 'white_other_perc',
 'white_roma_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_ethnicity_lsoa2021_gdb_df['dominant_detailed_ethnic_group'] = census2021_ethnicity_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_ethnicity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,dominant_ethnic_group,dominant_detailed_ethnic_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,158,11,56,68,1181,10.719132,0.746269,3.799186,4.613297,80.122117,4,66,49,37,2,8,3,0,18,28,8,2,46,22,822,0,32,316,11,1474,0.271370,4.477612,3.324288,2.510176,0.135685,0.542741,0.203528,0.000000,1.221167,1.899593,0.542741,0.135685,3.120760,1.492537,55.766621,0.0,2.170963,21.438263,0.746269,white,white_british
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,181,11,60,50,1084,13.059163,0.793651,4.329004,3.607504,78.210678,7,102,37,32,3,9,2,0,28,26,4,2,37,13,734,0,42,302,6,1386,0.505051,7.359307,2.669553,2.308802,0.216450,0.649351,0.144300,0.000000,2.020202,1.875902,0.288600,0.144300,2.669553,0.937951,52.958153,0.0,3.030303,21.789322,0.432900,white,white_british
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,153,56,101,107,1195,9.491315,3.473945,6.265509,6.637717,74.131514,11,52,37,50,3,34,15,7,44,36,14,7,89,18,815,0,33,343,4,1612,0.682382,3.225806,2.295285,3.101737,0.186104,2.109181,0.930521,0.434243,2.729529,2.233251,0.868486,0.434243,5.521092,1.116625,50.558313,0.0,2.047146,21.277916,0.248139,white,white_british
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,354,119,79,125,424,32.152589,10.808356,7.175295,11.353315,38.510445,263,35,24,31,1,77,27,15,32,16,11,20,101,24,255,0,17,148,4,1101,23.887375,3.178928,2.179837,2.815622,0.090827,6.993642,2.452316,1.362398,2.906449,1.453224,0.999092,1.816530,9.173479,2.179837,23.160763,0.0,1.544051,13.442325,0.363306,white,"asian_bangladeshi, white_british"
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E05014066,Northbury,E09000002,Barking and Dagenham,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.

# 9. Upload to geodatabase

In [30]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_ethnicity"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [31]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_ethnicity_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_ethnicity_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [32]:
# Publish the GeoDataFrame to PostGIS
census2021_ethnicity_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_ethnicity
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_ethnicity
